In [33]:
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import TypedDict,Annotated

In [34]:
# RAG Implementation
# Load the documents PDF
loader = PyPDFLoader(r'C:\Users\Mohinesh\Desktop\GitRepositories\gen-ai-projects\06-ChatBot-RAG\sample.pdf')
docs = loader.load()
docs




[Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2026-08-18T15:36:01+00:00', 'moddate': '2026-08-18T15:36:01+00:00', 'source': 'C:\\Users\\Mohinesh\\Desktop\\GitRepositories\\gen-ai-projects\\06-ChatBot-RAG\\sample.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='SAMPLE PDF — 5 PAGES\nPage 1 of 5\nSAMPLE PDF DOCUMENT\n5 Pages · Black & White Text · QwikPDF\nDOCUMENT INFORMATION\nTotal Pages:\n5 pages\nFormat:\nPDF 1.7 (Portable Document Format)\nContent:\nBlack & white text — no images, no colour\nFonts:\nHelvetica, Helvetica-Bold, Courier\nCreated By:\nQwikPDF Browser Generator\nWebsite:\nhttps://qwikpdf.com\nGenerated:\nTue, 18 Aug 2026 15:36:01 GMT\nPurpose:\nShort document testing, upload form validation\nSection 1: Introduction & Overview\nLorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt\nut labore et dolore magna a

In [35]:
# Split the PDF
splitter = RecursiveCharacterTextSplitter()
chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2026-08-18T15:36:01+00:00', 'moddate': '2026-08-18T15:36:01+00:00', 'source': 'C:\\Users\\Mohinesh\\Desktop\\GitRepositories\\gen-ai-projects\\06-ChatBot-RAG\\sample.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='SAMPLE PDF — 5 PAGES\nPage 1 of 5\nSAMPLE PDF DOCUMENT\n5 Pages · Black & White Text · QwikPDF\nDOCUMENT INFORMATION\nTotal Pages:\n5 pages\nFormat:\nPDF 1.7 (Portable Document Format)\nContent:\nBlack & white text — no images, no colour\nFonts:\nHelvetica, Helvetica-Bold, Courier\nCreated By:\nQwikPDF Browser Generator\nWebsite:\nhttps://qwikpdf.com\nGenerated:\nTue, 18 Aug 2026 15:36:01 GMT\nPurpose:\nShort document testing, upload form validation\nSection 1: Introduction & Overview\nLorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt\nut labore et dolore magna a

In [36]:
# Store in the Vector store by generating embeddings
embedding_model = GoogleGenerativeAIEmbeddings(model='gemini-embedding-2')
vector_store = FAISS.from_documents(chunks,embedding_model)

In [37]:
list(vector_store.docstore._dict.values())

[Document(id='6868fa73-1b8d-44ae-8ecb-01d14d3b8cd9', metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2026-08-18T15:36:01+00:00', 'moddate': '2026-08-18T15:36:01+00:00', 'source': 'C:\\Users\\Mohinesh\\Desktop\\GitRepositories\\gen-ai-projects\\06-ChatBot-RAG\\sample.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='SAMPLE PDF — 5 PAGES\nPage 1 of 5\nSAMPLE PDF DOCUMENT\n5 Pages · Black & White Text · QwikPDF\nDOCUMENT INFORMATION\nTotal Pages:\n5 pages\nFormat:\nPDF 1.7 (Portable Document Format)\nContent:\nBlack & white text — no images, no colour\nFonts:\nHelvetica, Helvetica-Bold, Courier\nCreated By:\nQwikPDF Browser Generator\nWebsite:\nhttps://qwikpdf.com\nGenerated:\nTue, 18 Aug 2026 15:36:01 GMT\nPurpose:\nShort document testing, upload form validation\nSection 1: Introduction & Overview\nLorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tem

In [38]:
# Create Retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 1})

In [39]:
def llm_call(state):
    # Set title only once
    if not state.get("title"):
        title = state["messages"][-1].content[:30]
        if len(state["messages"][-1].content) > 30:
            title += "..."
        state["title"] = title
    response = model_with_tools.invoke(state["messages"])
    return {
        "messages" : [response],
        "title" : state["title"]
    }

In [40]:
model = ChatGoogleGenerativeAI(model='gemini-3.5-flash-lite')

# Create Tools
search_tool = DuckDuckGoSearchRun(region="us-en")

@tool
def retrieve_data(query)->dict:
    """Based on the query documents will be retrieved from the vector store"""
    return {
        'query':query,
        'results':retriever.invoke('what is background')
            }

@tool
def calculator(firstNumber:float , secondNumber:float,operation:str)->dict:
    """ Perform the mathematical addition operation for the given two numbers"""
    if  operation == "add":
        return {
            "firstNumber":firstNumber,
            "secondNumber":secondNumber,
            "operation":operation,
            "result":firstNumber + secondNumber
            }

tools = [search_tool,calculator,retrieve_data]
model_with_tools = model.bind_tools(tools)
tool_node = ToolNode(tools)

In [41]:
class ChatState(TypedDict):
    messages : Annotated[list,add_messages]
    title : str

In [42]:
graph = StateGraph(ChatState)
graph.add_node('llmcall',llm_call)
graph.add_node('tools',tool_node)
graph.add_edge(START,'llmcall')
graph.add_conditional_edges('llmcall',tools_condition)
graph.add_edge('tools','llmcall')

In [43]:
chatworkflow = graph.compile()

In [44]:
chatworkflow.invoke({'messages':['Tell me about the background of the uploaded pdf']})

{'messages': [HumanMessage(content='Tell me about the background of the uploaded pdf', additional_kwargs={}, response_metadata={}, id='7bf157df-d258-4576-b293-f8978d93f3e3'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'retrieve_data', 'arguments': '{"query": "background of the document"}'}, '__gemini_function_call_thought_signatures__': {'call_124211': 'El4KXAERTTIPrsY266qfqsQAGFTXYJ81uP2Og9bkHwz/Kn07QnRo2r3l3shlLZfN3abikPN+I1ZD0Yk5pBPFXtARGRiZa59di/Iou+SXHQ2W/tfOaoOOlFI3fgWgbWip'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a015a1-701f-7a40-970a-5113da1e2260-0', tool_calls=[{'name': 'retrieve_data', 'args': {'query': 'background of the document'}, 'id': 'call_124211', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 208, 'output_tokens': 19, 'total_tokens': 227, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(c